In [1]:
import os
import torch
import torch.nn as nn
import pickle
import numpy as np
import base64
from io import BytesIO
from PIL import Image
from tqdm import tqdm
import pandas as pd
import kagglehub
import itertools
from torch.utils.data import DataLoader
import torchvision.transforms as T
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
import seaborn as sns

# If you have the explainers_lib installed locally, keep this. 
# Otherwise, I have included a simple grouping function below.
import sys
sys.path.append(os.path.abspath('../src'))
from explainers_lib.ensemble import cfs_group_by_original_data
from explainers_lib.aggregators import IdealPoint, TOPSIS, BalancedPoint, ParetoMeanPoint, RandomPoint
from explainers_lib.model import TorchModel
from explainers_lib.datasets import Dataset as ExplainersDataset

# Force CPU
device = torch.device('cpu')
print(f'Using device: {device}')

# ---------------------------------------------------------
# 1. MINIMAL MODEL DEFINITIONS (Decoder Only)
# ---------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_c), nn.LeakyReLU(0.2)
        )
    def forward(self, x): return self.block(x)

class TransConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(out_c), nn.LeakyReLU(0.2)
        )
    def forward(self, x): return self.block(x)

class Encoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128),
            ConvBlock(128, 256), ConvBlock(256, 512), ConvBlock(512, 512),
            nn.Flatten(), nn.Linear(512 * 2 * 2, latent_dim), nn.Tanh()
        )
    def forward(self, x): return self.encoder(x)

class Decoder(nn.Module):
    def __init__(self, latent_dim=128):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(latent_dim, 512 * 2 * 2), nn.Tanh())
        self.decoder = nn.Sequential(
            TransConvBlock(512, 512), TransConvBlock(512, 512),
            TransConvBlock(512, 256), TransConvBlock(256, 128),
            TransConvBlock(128, 64), TransConvBlock(64, 32),
            nn.ConvTranspose2d(32, 3, kernel_size=3, stride=1, padding=1), nn.Tanh()
        )
    def forward(self, z):
        x = self.fc(z).view(-1, 512, 2, 2)
        return self.decoder(x)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.3),
            nn.Conv2d(64, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Dropout(0.3)
        )
        flatten_dim = 32 * 32 * 32
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(flatten_dim, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

class LatentToClassPipeline(nn.Module):
    __annotations__ = {}
    def __init__(self, decoder, cnn):
        super().__init__()
        self.decoder = decoder
        self.cnn = cnn
    def forward(self, x):
        decoded = self.decoder(x) * 0.5
        return self.cnn(decoded)

encoder = Encoder(latent_dim=128).to(device)
decoder = Decoder(latent_dim=128).to(device)
weights = torch.load('autoencoder_celeba_best.pth', map_location=device)
encoder.load_state_dict({k[len('encoder.'):]: v for k, v in weights.items() if k.startswith('encoder.')})
decoder.load_state_dict({k[len('decoder.'):]: v for k, v in weights.items() if k.startswith('decoder.')})
encoder.eval(); decoder.eval()

cnn = CNN()

cnn_weights = torch.load('models/torch_cnn_celeba_paper_smile_only.pth', map_location=device)

target_indices = [0, 1]

cnn_weights['classifier.4.weight'] = cnn_weights['classifier.4.weight'][target_indices]
cnn_weights['classifier.4.bias'] = cnn_weights['classifier.4.bias'][target_indices]

cnn.load_state_dict(cnn_weights)
cnn.to(device).eval()

pipeline = LatentToClassPipeline(decoder, cnn).to(device)
scripted_pipeline = torch.jit.script(pipeline.to(device))
model = TorchModel(scripted_pipeline)

# ---------------------------------------------------------
# 2. LOAD SAVED COUNTERFACTUALS
# ---------------------------------------------------------
PICKLE_FILE_PATH = "results/evaluate_selectors_celeba_20260313_071238.pkl"

with open(PICKLE_FILE_PATH, "rb") as f:
    all_generated_cfes = pickle.load(f)

print(f"Successfully loaded {len(all_generated_cfes)} counterfactuals.")

# Group them by original instance
grouped_cfs = cfs_group_by_original_data(all_generated_cfes)

Using device: cpu
Successfully loaded 391 counterfactuals.


In [ ]:
# ---------------------------------------------------------
# Selectors
# ---------------------------------------------------------
path = kagglehub.dataset_download("jessicali9530/celeba-dataset")
img_dir = os.path.join(path, "img_align_celeba/img_align_celeba")
attr_path = os.path.join(path, "list_attr_celeba.csv")

df = pd.read_csv(attr_path)
df = df.rename(columns=lambda s: s.strip())
df["class"]  = df["Smiling"].apply(lambda x: 1 if x == 1 else 0)

train_df, test_df = train_test_split(df, test_size=1000, shuffle=True, random_state=42)
transform = T.Compose([T.Resize((128, 128)), T.ToTensor(), T.Normalize((0.5,)*3, (0.5,)*3)])

class Celeb4ClassDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.df, self.img_dir, self.transform = dataframe, img_dir, transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row["image_id"])).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, torch.tensor(row["class"], dtype=torch.long)

print("Data loader")
train_dl = DataLoader(Celeb4ClassDataset(train_df, img_dir, transform), batch_size=64, shuffle=True)

def get_latent_dataset(dataloader, n_samples=5000):
    latent_list, labels_list = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(dataloader):
            latent = encoder(imgs.to(device)).cpu().numpy()
            latent_list.append(latent)
            labels_list.append(labels.numpy())
            if len(np.concatenate(latent_list)) >= n_samples: break
    
    X = np.concatenate(latent_list)[:n_samples]
    y = np.concatenate(labels_list)[:n_samples]
    feature_names = [f'l{i}' for i in range(128)]
    df_latent = pd.DataFrame(X, columns=feature_names)
    
    identity = ColumnTransformer([('num', 'passthrough', feature_names)], remainder='drop')
    identity.fit(df_latent)
    
    return ExplainersDataset(df_latent, y, features=feature_names, 
                             continuous_features=feature_names, preprocessor=identity)

print("Latent dataset")
latent_train_ds = get_latent_dataset(train_dl, n_samples=5000)

selectors = {
    # "Pareto": Pareto(),
    "RandomPoint": RandomPoint(),
    "IdealPoint": IdealPoint(),
    "BalancedPoint": BalancedPoint(),
    "TOPSIS": TOPSIS(k=1),
    "ParetoMeanPoint": ParetoMeanPoint(),
}

print("Selectors fit")
for name, s in tqdm(selectors.items()):
    s.fit(model, latent_train_ds)

Data loader
Latent dataset


  2%|▏         | 78/3150 [00:24<15:51,  3.23it/s]


Selectors fit


  0%|          | 0/5 [00:00<?, ?it/s]

Fitting RandomPoint
Fitting IdealPoint


...

 60%|██████    | 3/5 [03:40<02:35, 77.65s/it]

Fitting TOPSIS


 80%|████████  | 4/5 [05:28<01:29, 89.02s/it]

Fitting ParetoMeanPoint


100%|██████████| 5/5 [07:16<00:00, 87.23s/it]


In [34]:
selectors["RandomPoint"] = RandomPoint(seed = 42)

In [35]:
scores_map = dict()
results_by_selector = {name: [] for name in selectors}
for original_idx, cfs in tqdm(grouped_cfs.items()):
    scores_df = selectors["TOPSIS"].calculate_scores(cfs)
    for j, cf in enumerate(cfs):
        scores_map[cf.data.tobytes()] = scores_df.iloc[j].to_dict()
    
    for name, selector in selectors.items():
        selected = selector(cfs)
        results_by_selector[name].extend(map(lambda cf: cf.data.tobytes(), selected))

100%|██████████| 19/19 [00:27<00:00,  1.44s/it]


In [58]:
from datetime import datetime
from pickle import dump
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(f"results/evaluate_selectors_celeba_{timestamp}_scores.pkl", "wb") as f:
    dump({ "scores_map": scores_map, "results_by_selector": results_by_selector }, f)  